# Manual Run - ETL iSeller Pusat DWH

> Notebook untuk menjalankan pipeline ETL iSeller Pusat secara manual.
>
> **Flow:**
> 1. Backfill Fulfillment (detect unfulfilled orders → re-pull API → update DL)
> 2. Re-dump 8 DWH Summary Tables secara berurutan
> 3. Kirim email report
>
> Created by data-team

---
## CONFIGURATION

In [ ]:
import datetime
import sys
import os

# ===================== SET MODE =============================
#    True  = Otomatis (kemarin / current_date - 1)
#    False = Manual (isi START_DATE & END_DATE sendiri)
# ============================================================

AUTO_MODE = False

# ============================================================
#  MODE MANUAL: Isi tanggal di bawah (isi hanya jika AUTO_MODE = False)
# ============================================================

MANUAL_START_DATE = '2026-10-01'
MANUAL_END_DATE   = '2026-10-07'

# ===================== DRY RUN ==============================
#    True  = Backfill dijalankan penuh (baca DL + API iSeller) tapi TIDAK menulis apa pun:
#            details tidak di-dump, header tidak di-UPDATE, email & Teams tidak dikirim,
#            DWH tidak disentuh. Hasil (df_headers, df_details) ditampilkan di bagian DRY RUN RESULT.
#    False = Run asli (menulis Data Lake / DWH sesuai parameter di bawah)
# ============================================================

DRY_RUN = True

# ===================== SKIP DL CHECK ========================
#    True  = Skip backfill fulfillment (cek unfulfilled order)
#            (gunakan jika DL sudah di-clean manual,
#             langsung re-dump DWH saja + kirim report email)
#            -> setara run_mode = dwh_only di Airflow
#    False = Full flow (backfill fulfillment → re-dump DWH)
# ============================================================

SKIP_DL_CHECK = False

# ===================== SKIP DWH =============================
#    True  = Backfill Data Lake saja, DWH tidak disentuh
#            -> setara run_mode = datalake_only di Airflow
#    False = Lanjut re-dump DWH setelah backfill
# ============================================================

SKIP_DWH = False

# ============================================================

if SKIP_DL_CHECK and SKIP_DWH:
    raise ValueError("SKIP_DL_CHECK dan SKIP_DWH tidak boleh True bersamaan (tidak ada yang dijalankan).")
if DRY_RUN and SKIP_DL_CHECK:
    raise ValueError("DRY_RUN hanya untuk backfill (STEP 1). Set SKIP_DL_CHECK = False.")

if AUTO_MODE:
    yesterday = (datetime.date.today() - datetime.timedelta(days=1)).strftime("%Y-%m-%d")
    START_DATE = (datetime.date.today() - datetime.timedelta(days=9)).strftime("%Y-%m-%d")
    END_DATE   = yesterday
else:
    START_DATE = MANUAL_START_DATE
    END_DATE   = MANUAL_END_DATE

print(f"Mode      : {'AUTO (yesterday - 8 days window)' if AUTO_MODE else 'MANUAL'}")
print(f"Period    : {START_DATE} s/d {END_DATE}")
print(f"Dry run   : {DRY_RUN}")
print(f"Skip DL   : {SKIP_DL_CHECK}")
print(f"Skip DWH  : {SKIP_DWH}{'  (diabaikan: dry run selalu berhenti setelah backfill)' if DRY_RUN else ''}")

---
## LOAD CONFIG & IMPORT PIPELINE

In [ ]:
import importlib, importlib.util

def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dw = module_from_file("config_dw", "../../config/config_db_datawarehouse.py")

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'dags', 'iseller_pusat_fulfillment', 'pipelines'))
sys.path.insert(0, os.path.join(os.path.abspath('.'), 'pipelines'))
# `common.*` (db_helpers, teams_powerautomate_report) ada di <repo>/plugins. Di container Airflow sudah ada
# di PYTHONPATH; untuk Jupyter lokal ditambahkan di sini.
PLUGINS_DIR = os.path.abspath(os.path.join('..', '..', 'plugins'))
if os.path.isdir(PLUGINS_DIR) and PLUGINS_DIR not in sys.path:
    sys.path.insert(0, PLUGINS_DIR)

from etl_orchestrator import run_all

print("Config & pipeline loaded successfully.")

---
## RUN PIPELINE

In [ ]:
results = run_all(
    start_date=START_DATE,
    end_date=END_DATE,
    config_dw=config_dw,
    skip_dl_check=SKIP_DL_CHECK,
    skip_dwh=SKIP_DWH,
    dry_run=DRY_RUN,
)

---
## REVIEW RESULTS

In [ ]:
if not results:
    print("No results returned.")
elif results.get('_stopped'):
    # run_all berhenti sebelum / tanpa redump DWH (dry_run, all fulfilled, hub sync required, datalake_only)
    print(f"Pipeline berhenti: {results.get('_reason')}")
    if results.get('_error'):
        print(f"Error: {results['_error']}")
    stats = results.get('backfill') or {}
    if stats:
        print("\nBackfill stats:")
        for k in ['total_fulfilled', 'fulfilled_amount', 'total_unfulfilled', 'unfulfilled_amount',
                  'total_details_dumped', 'details_skipped_existing']:
            if k in stats:
                v = stats[k]
                print(f"  {k:<26}: {v:,.0f}" if isinstance(v, (int, float)) else f"  {k:<26}: {v}")
else:
    print(f"{'Table':<55} {'Status':<10} {'Deleted':>10} {'Dumped':>10}")
    print("-" * 90)
    for table_name, r in results.items():
        if r['status'] == 'SUCCESS':
            print(f"{table_name:<55} {r['status']:<10} {r['deleted']:>10,} {r['dumped']:>10,}")
        else:
            print(f"{table_name:<55} {r['status']:<10} Error: {r['error']}")

---
## DRY RUN RESULT

> Hanya terisi kalau `DRY_RUN = True`. Data di bawah adalah **persis** yang akan ditulis kalau run asli:
> - `df_headers` — header (dari API) order yang jadi fulfilled. Di Data Lake yang di-UPDATE hanya kolom
>   `fulfillment_status = 'fulfilled'` dan `sales_type`.
> - `df_details` — details yang akan di-append ke `transactions_items_iseller_pusat`.
> - `df_details_skipped` — details order yang sudah punya details di Data Lake (tidak di-append ulang).
> - `df_unfulfilled_dl` — snapshot order unfulfilled dari Data Lake (input backfill).

In [ ]:
import pandas as pd

frames = ((results or {}).get('backfill') or {}).get('frames', {}) if (results or {}).get('_reason') == 'dry_run' else {}
if not frames:
    print("Tidak ada hasil dry run (DRY_RUN = False, atau tidak ada order unfulfilled).")

df_unfulfilled_dl  = frames.get('df_unfulfilled_dl', pd.DataFrame())
df_headers         = frames.get('df_headers', pd.DataFrame())
df_details         = frames.get('df_details', pd.DataFrame())
df_details_skipped = frames.get('df_details_skipped', pd.DataFrame())

print(f"df_unfulfilled_dl  : {len(df_unfulfilled_dl):,} row ({df_unfulfilled_dl['order_id'].nunique() if len(df_unfulfilled_dl) else 0:,} order)")
print(f"df_headers         : {len(df_headers):,} row ({df_headers['order_id'].nunique() if len(df_headers) else 0:,} order fulfilled)")
print(f"df_details         : {len(df_details):,} row (akan di-append)")
print(f"df_details_skipped : {len(df_details_skipped):,} row (order sudah punya details di DL)")
if len(df_details):
    print(f"\norder_detail_id dobel di df_details: {int(df_details.duplicated(['order_detail_id', 'product_id', 'transactions_type']).sum())}"
          "  (non-0 wajar hanya untuk combo dengan komponen identik)")

In [ ]:
# Header order yang akan jadi fulfilled (yang di-UPDATE di DL: fulfillment_status + sales_type)
print(df_headers['sales_type'].value_counts().to_string() if len(df_headers) else "-")
df_headers

In [ ]:
# Details yang akan di-append ke transactions_items_iseller_pusat
print(df_details['type'].value_counts().to_string() if len(df_details) else "-")
df_details

In [ ]:
# Rekonsiliasi per order: total details vs total_amount header (transaksi sale)
if len(df_headers) and len(df_details):
    h = (df_headers[df_headers['transactions_type'] == 'sale']
         .drop_duplicates(['order_id', 'transaction_id'])
         .assign(total_amount=lambda x: pd.to_numeric(x['total_amount'], errors='coerce'))
         .groupby('order_id').agg(order_reference=('order_reference', 'first'),
                                  sales_type=('sales_type', 'first'),
                                  header_total=('total_amount', 'sum')))
    d = (df_details[df_details['transactions_type'] == 'sale']
         .assign(total_amount=lambda x: pd.to_numeric(x['total_amount'], errors='coerce'))
         .groupby('order_id').agg(details_total=('total_amount', 'sum'), n_rows=('order_detail_id', 'size')))
    recon = h.join(d, how='inner')
    recon['ratio'] = (recon['details_total'] / recon['header_total']).astype(float).round(4)
    off = recon[(recon['ratio'] - 1).abs() > 0.01]
    print(f"Total header  : {recon['header_total'].sum():,.2f}")
    print(f"Total details : {recon['details_total'].sum():,.2f}  ({recon['details_total'].sum() / recon['header_total'].sum():.4f}x)")
    print(f"Order meleset > 1%: {len(off)} dari {len(recon)}")
    display(off.sort_values('ratio') if len(off) else recon.head(20))
else:
    print("-")